# Exploração dos Dados Brutos: NYC TLC Trip Records

Notebook de análise exploratória (EDA) dos arquivos parquet brutos do NYC TLC, executado **antes** do desenvolvimento do pipeline.

O objetivo é entender schema, qualidade, distribuições e anomalias de cada tipo de táxi para embasar as decisões de design da Medallion Architecture.

**Tipos analisados:** Yellow, Green, FHV (For-Hire Vehicle), FHVHV (High Volume FHV - Uber/Lyft)  
**Período amostrado:** Janeiro/2023 (amostra representativa antes de processar os 5 meses)

In [0]:
import sys

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root = "/Workspace" + "/".join(notebook_path.split("/")[:-2])

sys.path.insert(0, project_root)
print(f"Project root: {project_root}")

Project root: /Workspace/Users/ewelimdsb1@gmail.com/NYC_taxi_pipeline


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

RAW_BASE = "/Volumes/nyc_taxi/main/data/raw"

def read_raw(taxi_type: str, year: int = 2023, month: int = 1) -> DataFrame:
    path = f"{RAW_BASE}/{taxi_type}/{taxi_type}_tripdata_{year}-{month:02d}.parquet"
    return spark.read.parquet(path)

yellow = read_raw("yellow")
green  = read_raw("green")
fhv    = read_raw("fhv")
fhvhv  = read_raw("fhvhv")

## 1. Schemas: O que cada fonte entrega?

Primeira pergunta: quais colunas existem, com quais tipos, e há sobreposição entre os tipos de táxi?

In [0]:
print("=== Yellow ===")
yellow.printSchema()

=== Yellow ===
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [0]:
print("=== Green ===")
green.printSchema()

=== Green ===
root
 |-- VendorID: long (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: integer (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- trip_type: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [0]:
print("=== FHV (For-Hire Vehicle) ===")
fhv.printSchema()

=== FHV (For-Hire Vehicle) ===
root
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropOff_datetime: timestamp_ntz (nullable = true)
 |-- PUlocationID: double (nullable = true)
 |-- DOlocationID: double (nullable = true)
 |-- SR_Flag: integer (nullable = true)
 |-- Affiliated_base_number: string (nullable = true)



In [0]:
print("=== FHVHV (High Volume FHV - Uber, Lyft) ===")
fhvhv.printSchema()

=== FHVHV (High Volume FHV - Uber, Lyft) ===
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: 

In [0]:
yellow_cols = set(yellow.columns)
green_cols  = set(green.columns)
fhv_cols    = set(fhv.columns)
fhvhv_cols  = set(fhvhv.columns)

print("=== Colunas exclusivas por tipo ===")
print(f"Só Yellow:  {yellow_cols - green_cols - fhv_cols - fhvhv_cols}")
print(f"Só Green:   {green_cols - yellow_cols - fhv_cols - fhvhv_cols}")
print(f"Só FHV:     {fhv_cols - yellow_cols - green_cols - fhvhv_cols}")
print(f"Só FHVHV:   {fhvhv_cols - yellow_cols - green_cols - fhv_cols}")
print()
print("=== Colunas comuns Yellow + Green ===")
print(yellow_cols & green_cols)
print()

for key_col in ["passenger_count", "total_amount"]:
    print(f"=== '{key_col}' existe? ===")
    for name, df in [("yellow", yellow), ("green", green), ("fhv", fhv), ("fhvhv", fhvhv)]:
        print(f"  {name}: {'SIM' if key_col in df.columns else 'NÃO'}")
    print()

=== Colunas exclusivas por tipo ===
Só Yellow:  {'tpep_pickup_datetime', 'tpep_dropoff_datetime'}
Só Green:   {'lpep_dropoff_datetime', 'trip_type', 'lpep_pickup_datetime', 'ehail_fee'}
Só FHV:     {'Affiliated_base_number', 'DOlocationID', 'dropOff_datetime', 'SR_Flag', 'PUlocationID'}
Só FHVHV:   {'shared_match_flag', 'dropoff_datetime', 'wav_match_flag', 'bcf', 'tolls', 'access_a_ride_flag', 'sales_tax', 'driver_pay', 'originating_base_num', 'request_datetime', 'shared_request_flag', 'trip_miles', 'tips', 'on_scene_datetime', 'trip_time', 'hvfhs_license_num', 'wav_request_flag', 'base_passenger_fare'}

=== Colunas comuns Yellow + Green ===
{'DOLocationID', 'mta_tax', 'payment_type', 'tip_amount', 'improvement_surcharge', 'tolls_amount', 'total_amount', 'extra', 'VendorID', 'RatecodeID', 'congestion_surcharge', 'trip_distance', 'store_and_fwd_flag', 'passenger_count', 'PULocationID', 'fare_amount'}

=== 'passenger_count' existe? ===
  yellow: SIM
  green: SIM
  fhv: NÃO
  fhvhv: NÃO


**Observação:** Yellow e Green possuem `passenger_count` e `total_amount`, compatíveis com as queries analíticas.
FHV e FHVHV **não possuem essas colunas** na mesma estrutura, o que determina que vão chegar apenas até o Bronze na nossa modelagem.

## 2. Volume de Dados

In [0]:
for name, df in [("yellow", yellow), ("green", green), ("fhv", fhv), ("fhvhv", fhvhv)]:
    print(f"{name:8s}: {df.count():>10,} registros em Jan/2023")

yellow  :  3,066,766 registros em Jan/2023
green   :     68,211 registros em Jan/2023
fhv     :  1,114,320 registros em Jan/2023
fhvhv   : 18,479,031 registros em Jan/2023


In [0]:
from functools import reduce

def volume_por_mes(taxi_type: str) -> DataFrame:
    rows = []
    for month in range(1, 6):
        try:
            n = read_raw(taxi_type, month=month).count()
            rows.append((taxi_type, month, n))
        except Exception as e:
            print(f"  WARN {taxi_type} {month:02d}: {e}")
    return spark.createDataFrame(rows, ["tipo", "mes", "registros"])

volume = reduce(
    lambda a, b: a.union(b),
    [volume_por_mes(t) for t in ["yellow", "green", "fhv", "fhvhv"]]
)
volume.orderBy("tipo", "mes").show(25)

+------+---+---------+
|  tipo|mes|registros|
+------+---+---------+
|   fhv|  1|  1114320|
|   fhv|  2|  1110797|
|   fhv|  3|  1328242|
|   fhv|  4|  1246479|
|   fhv|  5|  1385826|
| fhvhv|  1| 18479031|
| fhvhv|  2| 17960971|
| fhvhv|  3| 20413539|
| fhvhv|  4| 19144903|
| fhvhv|  5| 19847676|
| green|  1|    68211|
| green|  2|    64809|
| green|  3|    72044|
| green|  4|    65392|
| green|  5|    69174|
|yellow|  1|  3066766|
|yellow|  2|  2913955|
|yellow|  3|  3403766|
|yellow|  4|  3288250|
|yellow|  5|  3513649|
+------+---+---------+



## 3. Inconsistência de Tipos entre Meses

Problema conhecido do NYC TLC: o schema pode variar entre meses; colunas que eram `INT` em Janeiro podem ser `DOUBLE` em Maio, por exemplo. 
Isso causa falhas silenciosas ou erros de merge no Delta se não tratado na camada Bronze.

In [0]:
yellow_jan = read_raw("yellow", month=1)
yellow_mai = read_raw("yellow", month=5)

jan_types = {f.name: str(f.dataType) for f in yellow_jan.schema.fields}
mai_types = {f.name: str(f.dataType) for f in yellow_mai.schema.fields}

print("=== Colunas com tipo diferente entre Janeiro e Maio (Yellow) ===")
divergencias = {
    col: (jan_types[col], mai_types.get(col, "AUSENTE"))
    for col in jan_types
    if jan_types[col] != mai_types.get(col)
}
if divergencias:
    for col, (t1, t2) in divergencias.items():
        print(f"  {col:35s}  Jan={t1}  Mai={t2}")
else:
    print("  Nenhuma divergência encontrada")

=== Colunas com tipo diferente entre Janeiro e Maio (Yellow) ===
  VendorID                             Jan=LongType()  Mai=IntegerType()
  passenger_count                      Jan=DoubleType()  Mai=LongType()
  RatecodeID                           Jan=DoubleType()  Mai=LongType()
  PULocationID                         Jan=LongType()  Mai=IntegerType()
  DOLocationID                         Jan=LongType()  Mai=IntegerType()
  airport_fee                          Jan=DoubleType()  Mai=AUSENTE


**Conclusão:** se usarmos schema explícito na Bronze, os meses com tipo diferente vão falhar ou criar tabelas incompatíveis.
A solução é ler sem schema e normalizar todos os tipos inteiros para `Double` antes de gravar no Delta. Isso garante
que o `union` de partições diferentes na Silver não quebre.

## 4. Análise de Nulos

Quais colunas têm nulos? Em que proporção? Isso define quais filtros aplicar na Silver ou na Gold.

In [0]:
def null_report(df: DataFrame, name: str):
    total = df.count()
    print(f"\n=== Nulos em {name} (total: {total:,}) ===")
    null_counts = df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()
    for col, nulls in sorted(null_counts.items(), key=lambda x: -x[1]):
        if nulls > 0:
            pct = 100 * nulls / total
            print(f"  {col:40s}: {nulls:>8,}  ({pct:5.1f}%)")

null_report(yellow, "Yellow Jan/2023")
null_report(green,  "Green Jan/2023")
null_report(fhv,   "FHV Jan/2023")
null_report(fhvhv, "FHVHV Jan/2023")


=== Nulos em Yellow Jan/2023 (total: 3,066,766) ===
  passenger_count                         :   71,743  (  2.3%)
  RatecodeID                              :   71,743  (  2.3%)
  store_and_fwd_flag                      :   71,743  (  2.3%)
  congestion_surcharge                    :   71,743  (  2.3%)
  airport_fee                             :   71,743  (  2.3%)

=== Nulos em Green Jan/2023 (total: 68,211) ===
  ehail_fee                               :   68,211  (100.0%)
  trip_type                               :    4,334  (  6.4%)
  store_and_fwd_flag                      :    4,324  (  6.3%)
  RatecodeID                              :    4,324  (  6.3%)
  passenger_count                         :    4,324  (  6.3%)
  payment_type                            :    4,324  (  6.3%)
  congestion_surcharge                    :    4,324  (  6.3%)

=== Nulos em FHV Jan/2023 (total: 1,114,320) ===
  SR_Flag                                 : 1,114,320  (100.0%)
  PUlocationID              

## 5. Datas: Registros Corrompidos

NYC TLC tem um problema conhecido: motoristas inserem datas erradas no sistema (ex: 2001, 2087).
Vamos identificar a extensão do problema antes de decidir onde filtrar.

In [0]:
print("=== Distribuição de anos em tpep_pickup_datetime (Yellow) ===")
(yellow.withColumn("pickup_year", F.year("tpep_pickup_datetime"))
      .groupBy("pickup_year").count()
      .orderBy("pickup_year")
      .show())

=== Distribuição de anos em tpep_pickup_datetime (Yellow) ===
+-----------+-------+
|pickup_year|  count|
+-----------+-------+
|       2008|      2|
|       2022|     36|
|       2023|3066728|
+-----------+-------+



In [0]:
print("=== Distribuição de anos em lpep_pickup_datetime (Green) ===")
(green.withColumn("pickup_year", F.year("lpep_pickup_datetime"))
     .groupBy("pickup_year").count()
     .orderBy("pickup_year")
     .show(30))

=== Distribuição de anos em lpep_pickup_datetime (Green) ===
+-----------+-----+
|pickup_year|count|
+-----------+-----+
|       2009|    1|
|       2022|    2|
|       2023|68208|
+-----------+-----+



**Conclusão:** os registros corrompidos existem mas são residuais (< 0.1%).
São erros de input; não há sentido analítico em processar viagens de 2008.
A decisão de onde filtrar (Silver vs Gold) impacta rastreabilidade: filtrar só na Gold preserva os dados na Silver para auditoria.

## 6. `total_amount`: Outliers e Valores Negativos

In [0]:
print("=== Estatísticas de total_amount (Yellow) ===")
yellow.select(
    F.min("total_amount").alias("min"),
    F.percentile_approx("total_amount", 0.01).alias("p1"),
    F.percentile_approx("total_amount", 0.25).alias("p25"),
    F.percentile_approx("total_amount", 0.50).alias("mediana"),
    F.percentile_approx("total_amount", 0.75).alias("p75"),
    F.percentile_approx("total_amount", 0.99).alias("p99"),
    F.max("total_amount").alias("max"),
    F.avg("total_amount").alias("media"),
).show()

=== Estatísticas de total_amount (Yellow) ===
+------+---+----+-------+----+------+------+------------------+
|   min| p1| p25|mediana| p75|   p99|   max|             media|
+------+---+----+-------+----+------+------+------------------+
|-751.0|5.5|15.4|  20.16|28.7|101.94|1169.4|27.020383107156004|
+------+---+----+-------+----+------+------+------------------+



In [0]:
neg = yellow.filter(F.col("total_amount") < 0)
print(f"Registros com total_amount negativo: {neg.count():,}")
neg.select("total_amount", "tpep_pickup_datetime", "payment_type").show(10)

zero = yellow.filter(F.col("total_amount") == 0)
print(f"Registros com total_amount = 0: {zero.count():,}")

Registros com total_amount negativo: 25,204
+------------+--------------------+------------+
|total_amount|tpep_pickup_datetime|payment_type|
+------------+--------------------+------------+
|       -10.1| 2023-01-01 00:28:29|           4|
|       -14.3| 2023-01-01 00:20:18|           4|
|       -30.4| 2023-01-01 00:52:22|           4|
|       -10.1| 2023-01-01 00:06:39|           2|
|       -12.2| 2023-01-01 00:34:39|           4|
|        -5.5| 2023-01-01 00:26:51|           3|
|       -12.2| 2023-01-01 00:02:59|           4|
|      -55.15| 2023-01-01 00:40:02|           4|
|       -20.6| 2023-01-01 00:25:04|           4|
|        -8.0| 2023-01-01 00:50:09|           4|
+------------+--------------------+------------+
only showing top 10 rows
Registros com total_amount = 0: 568


In [0]:
print("=== Outliers altos (total_amount > 500) ===")
outliers = yellow.filter(F.col("total_amount") > 500)
(outliers
      .select("total_amount", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count")
      .orderBy(F.desc("total_amount"))
      .show(10))
print(f"Registros com total_amount > 500: {outliers.count():,}")

=== Outliers altos (total_amount > 500) ===
+------------+--------------------+---------------------+---------------+
|total_amount|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|
+------------+--------------------+---------------------+---------------+
|      1169.4| 2023-01-24 12:43:44|  2023-01-24 15:41:02|            1.0|
|      1000.0| 2023-01-09 16:17:32|  2023-01-09 16:20:41|            1.0|
|       901.0| 2023-01-30 13:17:33|  2023-01-30 13:17:48|            1.0|
|       751.0| 2023-01-30 13:23:56|  2023-01-30 13:24:08|            1.0|
|       705.6| 2023-01-30 16:17:35|  2023-01-31 08:33:24|            2.0|
|       667.1| 2023-01-04 20:11:27|  2023-01-04 22:15:19|            1.0|
|      656.85| 2023-01-08 08:15:40|  2023-01-08 08:15:58|            0.0|
|       651.0| 2023-01-22 23:24:55|  2023-01-22 23:25:02|            1.0|
|       626.0| 2023-01-10 00:55:47|  2023-01-10 00:56:53|            1.0|
|      614.45| 2023-01-29 14:46:13|  2023-01-29 16:18:22|           

## 7. `passenger_count`: Zeros, Nulos e Impacto na Média

In [0]:
print("=== Distribuição de passenger_count (Yellow) ===")
(yellow.groupBy("passenger_count").count()
      .orderBy("passenger_count")
      .show(20))

=== Distribuição de passenger_count (Yellow) ===
+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|           NULL|  71743|
|            0.0|  51164|
|            1.0|2261400|
|            2.0| 451536|
|            3.0| 106353|
|            4.0|  53745|
|            5.0|  42681|
|            6.0|  28124|
|            7.0|      6|
|            8.0|     13|
|            9.0|      1|
+---------------+-------+



In [0]:
total = yellow.count()
problematic = yellow.filter(F.col("passenger_count").isNull() | (F.col("passenger_count") == 0)).count()
print(f"passenger_count nulo ou zero: {problematic:,} de {total:,} ({100*problematic/total:.2f}%)")
print()

avg_com = yellow.agg(F.avg("passenger_count")).collect()[0][0]
avg_sem = yellow.filter(F.col("passenger_count") > 0).agg(F.avg("passenger_count")).collect()[0][0]
print(f"Média com zeros/nulos incluídos: {avg_com:.4f}")
print(f"Média excluindo zeros/nulos:     {avg_sem:.4f}")

passenger_count nulo ou zero: 122,907 de 3,066,766 (4.01%)

Média com zeros/nulos incluídos: 1.3625
Média excluindo zeros/nulos:     1.3862


**Conclusão:** zeros distorcem a média para baixo.\
**Decisão:** filtro passenger_count > 0 aplicado APENAS na query analítica, não na Silver/Gold. Isso preserva os dados originais para análises que precisem deles.

## 8. Duração das Corridas: Viagens Impossíveis

In [0]:
yellow_dur = yellow.withColumn(
    "duracao_min",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
)

print("=== Distribuição de duração das corridas (minutos) - Yellow ===")
yellow_dur.select(
    F.min("duracao_min").alias("min"),
    F.percentile_approx("duracao_min", 0.01).alias("p1"),
    F.percentile_approx("duracao_min", 0.50).alias("mediana"),
    F.percentile_approx("duracao_min", 0.99).alias("p99"),
    F.max("duracao_min").alias("max"),
).show()

neg_dur = yellow_dur.filter(F.col("duracao_min") < 0).count()
print(f"Corridas com duração negativa (dropoff antes do pickup): {neg_dur:,}")

=== Distribuição de duração das corridas (minutos) - Yellow ===
+-----+---+------------------+----+------------------+
|  min| p1|           mediana| p99|               max|
+-----+---+------------------+----+------------------+
|-29.2|0.8|11.516666666666667|57.3|10029.183333333332|
+-----+---+------------------+----+------------------+

Corridas com duração negativa (dropoff antes do pickup): 3


## 9. VendorID: Fornecedores de Hardware

In [0]:
print("=== VendorID (Yellow) ===")
yellow.groupBy("VendorID").count().orderBy("VendorID").show()

print("=== VendorID (Green) ===")
green.groupBy("VendorID").count().orderBy("VendorID").show()

print("Legenda (TPEP/LPEP):")
print("  1 → Creative Mobile Technologies, LLC")
print("  2 → Curb Mobility, LLC")
print("  6 → Myle Technologies Inc")
print("  7 → Helix (Yellow/TPEP apenas)")

=== VendorID (Yellow) ===
+--------+-------+
|VendorID|  count|
+--------+-------+
|       1| 827367|
|       2|2239399|
+--------+-------+

=== VendorID (Green) ===
+--------+-----+
|VendorID|count|
+--------+-----+
|       1| 9343|
|       2|58868|
+--------+-----+

Legenda (TPEP/LPEP):
  1 → Creative Mobile Technologies, LLC
  2 → Curb Mobility, LLC
  6 → Myle Technologies Inc
  7 → Helix (Yellow/TPEP apenas)


## 10. FHV e FHVHV: Por que ficam apenas no Bronze?

FHV e FHVHV têm estrutura incompatível com as queries analíticas que precisamos de `passenger_count` e `total_amount`.

In [0]:
print("=== FHV: colunas disponíveis ===")
print(fhv.columns)

print("\n=== FHVHV: colunas disponíveis ===")
print(fhvhv.columns)

=== FHV: colunas disponíveis ===
['dispatching_base_num', 'pickup_datetime', 'dropOff_datetime', 'PUlocationID', 'DOlocationID', 'SR_Flag', 'Affiliated_base_number']

=== FHVHV: colunas disponíveis ===
['hvfhs_license_num', 'dispatching_base_num', 'originating_base_num', 'request_datetime', 'on_scene_datetime', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_miles', 'trip_time', 'base_passenger_fare', 'tolls', 'bcf', 'sales_tax', 'congestion_surcharge', 'airport_fee', 'tips', 'driver_pay', 'shared_request_flag', 'shared_match_flag', 'access_a_ride_flag', 'wav_request_flag', 'wav_match_flag']


In [0]:
monetary_keywords = ["amount", "fare", "charge", "fee", "tip", "toll", "surcharge"]

fhv_monetary   = [c for c in fhv.columns   if any(kw in c.lower() for kw in monetary_keywords)]
fhvhv_monetary = [c for c in fhvhv.columns if any(kw in c.lower() for kw in monetary_keywords)]

print(f"FHV - colunas monetárias:   {fhv_monetary if fhv_monetary else 'nenhuma'}")
print(f"FHVHV - colunas monetárias: {fhvhv_monetary}")

FHV - colunas monetárias:   nenhuma
FHVHV - colunas monetárias: ['base_passenger_fare', 'tolls', 'congestion_surcharge', 'airport_fee', 'tips']


**Conclusão:** FHVHV usa 'base_passenger_fare' em vez de 'total_amount'.
Para incluir FHVHV na Gold seria necessário harmonização explícita desta coluna. Não vamos levar esses dados para a Silver/Gold.

## 11. Resumo das Descobertas

| Problema encontrado | Impacto | Decisão no pipeline |
|---|---|---|
| Yellow e Green usam nomes diferentes para datas | Schema não unificável diretamente | Renomear para `pickup_datetime`/`dropoff_datetime` na Silver |
| Tipos INT vs DOUBLE variam entre meses (ex: `VendorID`) | `DELTA_FAILED_TO_MERGE_FIELDS` ao fazer union de partições | `_normalize_types()` na Bronze: cast de todos INT para Double |
| Registros com `pickup_datetime` em 2008/2087 | Partições `year=2008` na Silver, distorção em queries | Filtro `>= PIPELINE_START_DATE` aplicado na Gold |
| `total_amount` negativo ou zero | Comportamento esperado para `payment_type` 3 (No charge) e 4 (Dispute); não são erros | Filtro `total_amount > 0` aplicado apenas na Query 1 analítica; Silver e Gold preservam todos os registros |
| `passenger_count = 0` ou NULL | Distorce média de passageiros | Filtro aplicado apenas na Query 2 (não na Silver/Gold) |
| FHV não tem `passenger_count` nem `total_amount` | Incompatível com queries obrigatórias | Ingere Bronze apenas; não processa Silver/Gold |
| FHVHV usa `base_passenger_fare` em vez de `total_amount` | Requer harmonização não trivial | Bronze apenas; melhoria futura documentada no README |
| Green tem volume ~50× menor que Yellow | Partições pequenas (small files) | Aceito: `replaceWhere` justifica particionamento mesmo com arquivos pequenos |

### Schema canônico resultante (Silver/Gold)

| Coluna | Fonte Yellow | Fonte Green | Tipo final |
|---|---|---|---|
| `VendorID` | `VendorID` | `VendorID` | integer |
| `passenger_count` | `passenger_count` | `passenger_count` | integer |
| `total_amount` | `total_amount` | `total_amount` | double |
| `pickup_datetime` | `tpep_pickup_datetime` | `lpep_pickup_datetime` | timestamp |
| `dropoff_datetime` | `tpep_dropoff_datetime` | `lpep_dropoff_datetime` | timestamp |
| `taxi_type` | literal `'yellow'` | literal `'green'` | string |